# 1. Prepare Snowflake data for TFT feature experiments

All input data, including festive features, comes from the configured Snowflake table.
No Excel or CSV calendar is loaded. Point TABLE_NAME at the table containing the
updated features. Preserve the old table/snapshot if you want a legacy comparison.

Run this notebook first, then modelling_code_new.ipynb from the same working directory.
Each extraction has its own directory; old exports are never silently reused.
Only target, IDs and static attributes are downloaded per series. The future calendar
is downloaded once. No future sales targets are downloaded or used for training.

FEATURE_SET="engineered" requires all seven new fields; "legacy" requires the old
37 N/D fields. Use the SAME rows, snapshot, calendar policy and modelling settings
for both experiments. Save each resulting column_roles.json path for the model notebook.
Use ROW_FILTER_SQL to select Runner/A-class series only if that classification exists
in your source table and was known before the backtest cutoff.

In [ ]:
import os, sys, json, hashlib, gc
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import numpy as np
import pandas as pd
from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import Snowflake_configuration

TABLE_NAME = 'MOP_DATABASE.SOQ.DAILY_DATA_FOR_FORECASTING_WITH_FESTIVE_FEATURES'
FEATURE_SET = 'engineered'           # 'engineered' or 'legacy'
SOURCE_REVISION = 'festive_features_v2'  # Label the source snapshot/extraction.
ROW_FILTER_SQL = None               # Optional Snowflake SQL predicate, e.g. a known SKU tier.
CHUNK_SIZE = 20                     # Dealers per download; lower peak host RAM than 100.
HISTORY_START = '2023-04-01'
HISTORY_END = '2026-08-31'
CALENDAR_END = '2026-12-07'
FREQ = 'D'
time_col = 'CAL_DATE'
group_col = 'PARENT_DEALER_CODE_MODEL_FAMILY'
dealer_col = 'PARENT_DEALER_CODE'
target_col = 'NET_SALES'
BASE = Path.cwd()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:8]
EXPORT_DIR = BASE/'tft_exports'/f'{FEATURE_SET}_{RUN_ID}'

static_covariates = [
    'PARENT_DEALER_CODE','MODEL_FAMILY','MODEL_NAME','BRAKE_TYPE',
    'IGNITION_TYPE','WHEEL_TYPE','COLOUR','DEALER_CITY',
    'X_CITY_CATEGORY','ZONAL_OFFICE_NAME',
]
retained_festive = [
    'HARTALIK_TEEJ','GANESH_CHATURTHI','JANMASHTAMI','VISHWAKARMA_PUJA',
    'KARWA_CHAUTH','ONAM','HANUMAN_JAYANTI','AKSHYA_TRITIYA',
    'BUDDHA_PURNIMA','GANGA_DUSSEHRA','JAGANNATH_RATHYATRA',
    'GURU_PURNIMA','NAG_PANCHAMI','RAKSHA_BANDHAN','MARRIAGE_DAY',
]
new_festive = [
    'FESTIVE_DAYS_FROM_DIWALI','IS_NAVRATARI','IS_NAVRATARI_START',
    'IS_NAVRATARI_END','IS_DUSSEHRA','IS_PITRA_PAKSHA','IS_DHANTERAS',
]
old_festive = ([f'N-{i}' for i in range(16,0,-1)]+['N']+
               [f'N+{i}' for i in range(1,11)]+
               [f'D-{i}' for i in range(3,0,-1)]+['D']+
               [f'D+{i}' for i in range(1,7)])
assert FEATURE_SET in {'engineered','legacy'}
festive_covariates = retained_festive + (new_festive if FEATURE_SET=='engineered' else old_festive)
calendar_covariates = ['DOW_SIN','DOW_COS','IS_MONTH_END','IS_MONTH_START','DAYS_TO_MONTH_END']
future_covariates = festive_covariates + calendar_covariates

def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def atomic_json(path,payload):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(payload,indent=2,default=str)); os.replace(tmp,path)

In [ ]:
def validate_calendar(cal, feature_set):
    cal=cal.copy()
    cal[time_col]=pd.to_datetime(cal[time_col]).dt.normalize()
    cal=cal.sort_values(time_col).reset_index(drop=True)
    expected=pd.date_range(HISTORY_START,CALENDAR_END,freq='D')
    if cal[time_col].duplicated().any() or not pd.DatetimeIndex(cal[time_col]).equals(expected):
        raise ValueError('Calendar must contain exactly one row for every required date.')
    for c in festive_covariates:
        cal[c]=pd.to_numeric(cal[c],errors='raise')
    if not np.isfinite(cal[festive_covariates].to_numpy(float)).all():
        raise ValueError('Calendar contains missing or non-finite values.')
    binary=[c for c in festive_covariates if c!='FESTIVE_DAYS_FROM_DIWALI']
    if not cal[binary].isin([0,1]).all().all():
        raise ValueError('Expected binary festival flags; inspect the Snowflake values.')
    if feature_set=='engineered':
        dist=cal['FESTIVE_DAYS_FROM_DIWALI']
        if not dist.between(-20,10).all() or not np.equal(dist,np.rint(dist)).all():
            raise ValueError('FESTIVE_DAYS_FROM_DIWALI must be integer days clipped to [-20,10].')
    anchors=[]
    for year,g in cal.groupby(cal[time_col].dt.year):
        ns=g.loc[g['IS_NAVRATARI_START' if feature_set=='engineered' else 'N'].eq(1),time_col]
        ds=g.loc[g['FESTIVE_DAYS_FROM_DIWALI'].eq(0) if feature_set=='engineered' else g['D'].eq(1),time_col]
        if len(ns)!=1 or len(ds)!=1:
            raise ValueError(f'{year}: expected exactly one Navratri start and Diwali anchor.')
        nav,diwali=ns.iloc[0],ds.iloc[0]
        if feature_set=='engineered':
            expected_dist=(g[time_col]-diwali).dt.days.clip(-20,10)
            if not np.array_equal(g['FESTIVE_DAYS_FROM_DIWALI'],expected_dist):
                raise ValueError(f'{year}: reversed or inconsistent Diwali offsets.')
        anchors.append({'year':int(year),'navratri_start':str(nav.date()),'diwali':str(diwali.date())})
    dt=cal[time_col].dt
    cal['DOW_SIN']=np.sin(2*np.pi*dt.dayofweek/7)
    cal['DOW_COS']=np.cos(2*np.pi*dt.dayofweek/7)
    cal['IS_MONTH_END']=(dt.day>=dt.days_in_month-2).astype(float)
    cal['IS_MONTH_START']=(dt.day<=3).astype(float)
    cal['DAYS_TO_MONTH_END']=(dt.days_in_month-dt.day)/31.0
    cal[future_covariates]=cal[future_covariates].astype('float32')
    return cal[[time_col]+future_covariates],anchors

def validate_history_chunk(frame):
    frame=frame.copy()
    if frame[[time_col,group_col,target_col]+static_covariates].isna().any().any():
        raise ValueError('Missing target, date, key or static attribute; resolve upstream.')
    frame[time_col]=pd.to_datetime(frame[time_col]).dt.normalize()
    frame[group_col]=frame[group_col].astype(str)  # Preserve source keys exactly.
    for c in static_covariates:frame[c]=frame[c].astype(str)
    if frame.duplicated([group_col,time_col]).any():
        raise ValueError('Duplicate series/date rows; resolve upstream instead of summing silently.')
    y=pd.to_numeric(frame[target_col],errors='raise').to_numpy(dtype=np.float64)
    if not np.isfinite(y).all() or (y<0).any() or not np.equal(y,np.rint(y)).all():
        raise ValueError('NegativeBinomial requires non-negative integer targets. No clipping is applied.')
    if (y>2**24).any():raise ValueError('Counts exceed exact float32 integer range.')
    frame[target_col]=y.astype('float32')
    if frame.groupby(group_col)[static_covariates].nunique().gt(1).any().any():
        raise ValueError('A static attribute changes within a series. Fix the source definition.')
    return frame.sort_values([group_col,time_col]).reset_index(drop=True)

In [ ]:
# One shared calendar; history chunks contain no repeated festival columns.
# Credentials remain in your existing Snowflake_configuration module.
session=Session.builder.configs(Snowflake_configuration.ds1_role_json).create()
try:
    source=session.table(TABLE_NAME)
    names=[c.replace('"','') for c in source.columns]
    if len(names)!=len(set(names)):raise ValueError('Column names collide after removing quotes.')
    source=source.select(*[F.col(c).alias(c.replace('"','')) for c in source.columns])
    required=[time_col,group_col,target_col]+static_covariates+festive_covariates
    missing=sorted(set(required)-set(names))
    if missing:raise ValueError(f'Missing required Snowflake columns: {missing}')
    if ROW_FILTER_SQL:source=source.filter(F.sql_expr(ROW_FILTER_SQL))
    source=source.with_column(time_col,F.to_date(F.col(time_col)))
    source=source.filter((F.col(time_col)>=F.lit(HISTORY_START)) & (F.col(time_col)<=F.lit(CALENDAR_END)))
    aggregates=[F.count(F.lit(1)).alias('N_ROWS')]
    for i,c in enumerate(festive_covariates):
        aggregates += [F.min(F.col(c)).alias(f'F{i}_MIN'),F.max(F.col(c)).alias(f'F{i}_MAX'),F.count(F.col(c)).alias(f'F{i}_COUNT')]
    summary=source.group_by(time_col).agg(*aggregates).to_pandas()
    if summary.empty:raise ValueError('No rows returned from the source table.')
    cal=summary[[time_col]].copy()
    for i,c in enumerate(festive_covariates):
        if not ((summary[f'F{i}_MIN']==summary[f'F{i}_MAX']) & (summary[f'F{i}_COUNT']==summary.N_ROWS)).all():
            raise ValueError(f'{c} is missing or differs by dealer/SKU on the same date; a shared calendar is invalid.')
        cal[c]=summary[f'F{i}_MIN']
    cal,anchors=validate_calendar(cal,FEATURE_SET)
    history=source.filter(F.col(time_col)<=F.lit(HISTORY_END)).select(*([time_col,group_col,target_col]+static_covariates))
    dealers=sorted([r[dealer_col] for r in history.select(dealer_col).distinct().collect()],key=str)
    if not dealers:raise ValueError('No dealers in the extraction.')
    EXPORT_DIR.mkdir(parents=True,exist_ok=False)
    chunk_dir=EXPORT_DIR/'history';chunk_dir.mkdir()
    cal.to_parquet(EXPORT_DIR/'shared_calendar.parquet',index=False)
    index_parts=[];chunks=[]
    for j in range(0,len(dealers),CHUNK_SIZE):
        part=history.filter(F.col(dealer_col).isin(dealers[j:j+CHUNK_SIZE])).to_pandas()
        part=validate_history_chunk(part)
        name=f'chunk_{j//CHUNK_SIZE:04d}.parquet';path=chunk_dir/name
        tmp=path.with_suffix('.tmp.parquet');part.to_parquet(tmp,index=False);os.replace(tmp,path)
        idx=part[[group_col]+static_covariates].drop_duplicates(group_col).copy()
        idx['CHUNK_FILE']=name;index_parts.append(idx)
        chunks.append({'name':name,'rows':len(part),'sha256':file_hash(path)})
        print(f'{j//CHUNK_SIZE+1}/{(len(dealers)+CHUNK_SIZE-1)//CHUNK_SIZE}: {len(part):,} rows')
        del part;gc.collect()
    index=pd.concat(index_parts,ignore_index=True)
    if index[group_col].duplicated().any():raise ValueError('Series spans multiple dealer chunks; group key is not unique.')
    index=index.sort_values(group_col).reset_index(drop=True)
    index.to_parquet(EXPORT_DIR/'series_index.parquet',index=False)
    roles={'schema_version':2,'export_id':RUN_ID,'source_table':TABLE_NAME,'source_revision':SOURCE_REVISION,
           'row_filter_sql':ROW_FILTER_SQL,'feature_set':FEATURE_SET,'history_start':HISTORY_START,
           'history_end':HISTORY_END,'calendar_end':CALENDAR_END,'time_col':time_col,'group_col':group_col,
           'target_col':target_col,'freq':FREQ,'static_covariates':static_covariates,
           'festive_covariates':festive_covariates,'future_covariates':future_covariates,
           'anchors':anchors,'chunks':chunks,'calendar_sha256':file_hash(EXPORT_DIR/'shared_calendar.parquet'),
           'index_sha256':file_hash(EXPORT_DIR/'series_index.parquet')}
    atomic_json(EXPORT_DIR/'column_roles.json',roles)
    atomic_json(BASE/'active_tft_export.json',{'roles_path':str((EXPORT_DIR/'column_roles.json').resolve())})
    print(f'COMPLETE: {len(index):,} series; {len(future_covariates)} future covariates')
    print('Model notebook ROLES_PATH:',EXPORT_DIR/'column_roles.json')
finally:
    session.close()